# Занятие 4. ASR baseline на открытом аудио удмуртской речи

**Цель практики:** взять короткий открытый аудиофайл на удмуртском языке с Wikimedia Commons, прогнать маленькую ASR-модель и понять, получается ли осмысленная транскрипция.

Используем `openai/whisper-tiny` через Transformers. Это маленькая модель; она удобна для бесплатного Colab, но может плохо работать с языками, которых нет в её сильных режимах. Это и есть предмет анализа.

In [ ]:
!pip -q install transformers accelerate librosa soundfile pandas

In [ ]:
import os, re, json, textwrap, math, statistics, random, io
from pathlib import Path
import pandas as pd
import numpy as np
import requests

DATA_DIR = Path('/content/lowres_lab')
DATA_DIR.mkdir(exist_ok=True)

def show_df(df, n=10):
    display(df.head(n))

def save_artifact(name, obj):
    path = DATA_DIR / name
    if isinstance(obj, pd.DataFrame):
        obj.to_csv(path, index=False)
    else:
        path.write_text(str(obj), encoding='utf-8')
    print('saved:', path)

import librosa, soundfile as sf
from IPython.display import Audio
from transformers import pipeline

## 1. Скачиваем открытое аудио из Wikimedia Commons

In [ ]:
COMMONS_API = 'https://commons.wikimedia.org/w/api.php'
FILE_TITLE = 'File:Udmurt.ogg'
params = {
    'action': 'query',
    'titles': FILE_TITLE,
    'prop': 'imageinfo',
    'iiprop': 'url|mime|size|extmetadata',
    'format': 'json',
}
r = requests.get(COMMONS_API, params=params, timeout=30)
r.raise_for_status()
page = next(iter(r.json()['query']['pages'].values()))
info = page['imageinfo'][0]
audio_url = info['url']
print(audio_url)
print('license:', info.get('extmetadata', {}).get('LicenseShortName', {}).get('value'))

audio_path = DATA_DIR / 'udmurt.ogg'
audio_path.write_bytes(requests.get(audio_url, timeout=60).content)
Audio(str(audio_path))

## 2. Нормализуем аудио под ASR

In [ ]:
y, sr = librosa.load(audio_path, sr=16000, mono=True)
trimmed = y[: 30 * 16000]
wav_path = DATA_DIR / 'udmurt_16k.wav'
sf.write(wav_path, trimmed, 16000)
print('seconds:', round(len(trimmed) / 16000, 2))
Audio(str(wav_path))

## 3. Запускаем Whisper tiny

In [ ]:
asr = pipeline('automatic-speech-recognition', model='openai/whisper-tiny')
result = asr(str(wav_path), generate_kwargs={'task': 'transcribe'})
print(result['text'])
save_artifact('lesson04_asr_whisper_tiny.txt', result['text'])

## 4. Анализируем результат как baseline, а не как истину

In [ ]:
transcript = result['text']
analysis = {
    'chars': len(transcript),
    'tokens': len(transcript.split()),
    'cyrillic_share': len(re.findall(r'[А-Яа-яЁёӐ-ӿ]', transcript)) / max(len(transcript), 1),
    'latin_share': len(re.findall(r'[A-Za-z]', transcript)) / max(len(transcript), 1),
    'questions_for_human': [
        'на каком языке модель фактически распознала речь?',
        'есть ли совпадающие удмуртские слова?',
        'нужна ли ручная транскрипция хотя бы 30 секунд?',
        'какие шумы или особенности голоса мешают?',
    ],
}
print(json.dumps(analysis, ensure_ascii=False, indent=2))
save_artifact('lesson04_asr_analysis.json', json.dumps(analysis, ensure_ascii=False, indent=2))

## Вопросы для отчёта

1. Похожа ли транскрипция на речь в аудио?
2. Модель ошиблась языком или просто дала шумный текст?
3. Какой минимальный набор ручной разметки нужен для честной оценки?
4. Что должно быть в протоколе записи для будущих полевых данных?